# D101 — Generator Basics

This notebook introduces Python generators using small e-commerce examples.

## Learning goals

- Understand what a generator is and why it is useful.
- Distinguish collections, iterables, iterators, and generators.
- Consume generators with `for`, `next()`, and other iterator-based APIs.
- Compare an iterator class with a generator function.
- Build generators using both `yield` and generator expressions.
- Understand suspension, resumption, local state, and `StopIteration`.
- Understand that a normal generator does not run on another thread.

## 1. What is a generator?

A **generator** is a special kind of iterator that produces values one at a time. It is usually created in one of two ways:

1. A **generator function**, which contains one or more `yield` statements.
2. A **generator expression**, such as `(price * 0.9 for price in prices)`.

Calling a normal function executes it and eventually returns a result. Calling a generator function creates a generator object immediately; its body starts only when a consumer asks for a value.

Generators are useful when values are large, expensive, streamed, or may not all be required.

In [ ]:
def order_status_generator():
    print("GENERATOR: function body started")
    yield "placed"
    yield "packed"
    yield "shipped"
    print("GENERATOR: function body finished")

statuses = order_status_generator()
print("Generator object created:", statuses)
print("Notice: the function body has not started yet.")

## 2. Are lists and dictionaries generators?

No. Lists, tuples, sets, dictionaries, strings, and `range` objects are **iterables**, but they are not generators. They can create iterator objects when `iter()` is called.

Python does provide several built-in APIs whose results are lazy iterators:

- `enumerate(iterable)`
- `zip(first, second)`
- `map(function, iterable)`
- `filter(function, iterable)`
- `reversed(sequence)`

These objects behave like generators from the consumer's perspective, but technically they are built-in iterator types rather than generator objects created by `yield`.

A generator expression is a true generator and uses parentheses instead of list-comprehension brackets.

In [ ]:
products = ["book", "mouse", "keyboard"]
stock = {"book": 10, "mouse": 5}

list_iterator = iter(products)
dictionary_iterator = iter(stock)
built_in_lazy_iterator = enumerate(products, start=1)
true_generator = (product.upper() for product in products)

objects = {
    "list": products,
    "dictionary": stock,
    "list iterator": list_iterator,
    "enumerate object": built_in_lazy_iterator,
    "generator expression": true_generator,
}

for name, obj in objects.items():
    print(
        f"{name:20} type={type(obj).__name__:20} "
        f"iterator={iter(obj) is obj} generator={type(obj).__name__ == 'generator'}"
    )

For a more exact generator test, Python provides `inspect.isgenerator()`.

In [ ]:
import inspect

sample_generator = (value for value in range(3))
sample_map = map(str, range(3))

print("Generator expression:", inspect.isgenerator(sample_generator))
print("map object:", inspect.isgenerator(sample_map))
print("Both are iterators:", iter(sample_generator) is sample_generator,
      iter(sample_map) is sample_map)

## 3. Built-in lazy iterator examples

Although the following built-ins are not technically generator objects, they provide the same important lazy behavior: values are delivered on demand.

In [ ]:
product_names = ["book", "pen", "bag"]
prices = [200, 20, 800]

numbered_products = enumerate(product_names, start=1)
product_prices = zip(product_names, prices)
prices_with_tax = map(lambda price: round(price * 1.18, 2), prices)
expensive_prices = filter(lambda price: price >= 100, prices)

print("enumerate:", list(numbered_products))
print("zip:", list(product_prices))
print("map:", list(prices_with_tax))
print("filter:", list(expensive_prices))

Converting an iterator to `list` consumes it. Calling `list()` again on the same exhausted object produces an empty list.

In [ ]:
discounts = map(lambda price: price * 0.9, [100, 200])
print("First consumption:", list(discounts))
print("Second consumption:", list(discounts))

## 4. Generator expressions versus list comprehensions

A list comprehension calculates and stores all its results immediately. A generator expression calculates one result whenever it is requested.

In [ ]:
prices = [100, 200, 300]

discounted_list = [price * 0.9 for price in prices]
discounted_generator = (price * 0.9 for price in prices)

print("List result:", discounted_list)
print("Generator object:", discounted_generator)
print("First generated value:", next(discounted_generator))
print("Remaining generated values:", list(discounted_generator))

## 5. Generator and iterator consumer APIs

A generator **is an iterator**, so consumer code normally treats them alike. Both support:

- `iter(obj)`
- `next(obj)` and `next(obj, default)`
- `for item in obj`
- consumers such as `list()`, `tuple()`, `sum()`, `min()`, `max()`, and `any()`

Most of these operations consume values. A generator is usually single-use.

In [ ]:
def three_order_ids():
    yield "ORD-1"
    yield "ORD-2"
    yield "ORD-3"

order_ids = three_order_ids()

print("iter(generator) returns itself:", iter(order_ids) is order_ids)
print("next:", next(order_ids))
print("next:", next(order_ids))
print("next:", next(order_ids))
print("after exhaustion:", next(order_ids, "NO_MORE_ORDERS"))

### Same consumer, different producers

The consumer below knows only that it received an iterable. It works with a list, an iterator class, or a generator without needing separate loop logic.

In [ ]:
class OrderIdIterator:
    def __init__(self, count):
        self.current = 1
        self.count = count

    def __iter__(self):
        return self

    def __next__(self):
        if self.current > self.count:
            raise StopIteration
        order_id = f"ORD-{self.current}"
        self.current += 1
        return order_id

def order_id_generator(count):
    for number in range(1, count + 1):
        yield f"ORD-{number}"

def consume_orders(source, label):
    print(label)
    for order_id in source:
        print("  CONSUMER received", order_id)

consume_orders(["ORD-1", "ORD-2", "ORD-3"], "From a list")
consume_orders(OrderIdIterator(3), "From an iterator class")
consume_orders(order_id_generator(3), "From a generator")

## 6. Custom generator: next random odd number

This generator produces at most five random odd values. It prints when a value is requested, making lazy execution visible. A dedicated random-number object and seed make the demonstration repeatable.

To create an odd number, we generate an integer and transform it with `2 × n + 1`.

In [ ]:
import random

def random_odd_numbers(count=5, seed=42):
    """Yield at most five random odd numbers from 1 through 99."""
    if not 0 <= count <= 5:
        raise ValueError("count must be between 0 and 5")

    random_source = random.Random(seed)
    for position in range(1, count + 1):
        odd_number = 2 * random_source.randint(0, 49) + 1
        print(f"GENERATOR: fetched odd number #{position}: {odd_number}")
        yield odd_number

    print("GENERATOR: completed")

In [ ]:
odd_values = random_odd_numbers()
print("CONSUMER: generator created; no number has been generated yet")

print("CONSUMER next():", next(odd_values))
print("CONSUMER next():", next(odd_values))

print("CONSUMER: loop over the remaining values")
for value in odd_values:
    print("CONSUMER received:", value)

The first two `next()` calls and the later `for` loop all consume the same generator. The loop resumes from the third value because the generator remembers its paused state.

## 7. What happens behind `yield`?

Conceptually, Python handles a generator as a resumable function:

1. Calling the generator function creates a generator object. The function body has not started.
2. `next(generator)` begins or resumes the function.
3. Execution continues until a `yield value` statement.
4. That value is returned to the consumer, and the generator is **suspended**.
5. Its instruction position, local variables, and active loop/exception state are retained in a suspended execution frame.
6. The next request resumes immediately after that `yield`.
7. A normal function exit or `return` ends the generator and appears to the consumer as `StopIteration`.

`yield` is therefore both an output point and a pause point. Unlike `return`, it does not permanently discard the function's local state.

In [ ]:
def running_cart_total():
    total = 0
    print("Step 1: local total =", total)

    total += 100
    yield total

    # Resumption continues here; total is still 100.
    print("Step 2: resumed with local total =", total)
    total += 250
    yield total

    print("Step 3: resumed with local total =", total)
    total += 50
    yield total

cart_totals = running_cart_total()
print("Consumer request 1 ->", next(cart_totals))
print("Consumer request 2 ->", next(cart_totals))
print("Consumer request 3 ->", next(cart_totals))

### `return` inside a generator

A generator may use `return value` to finish. Internally, the value becomes the `.value` carried by `StopIteration`. A normal `for` loop hides this detail.

In [ ]:
def one_order_then_finish():
    yield "ORD-1"
    return "all orders processed"

single_order = one_order_then_finish()
print(next(single_order))

try:
    next(single_order)
except StopIteration as stop:
    print("Generator stopped with value:", stop.value)

## 8. Generator execution and the threading model

A normal generator is **not a thread**, background task, or parallel worker.

- The consumer and generator normally run on the **same thread**.
- Execution transfers synchronously: the consumer calls `next()`, the generator runs, and the consumer waits.
- At `yield`, the generator pauses and control returns to the consumer.
- Only one side is executing Python instructions at a time in this interaction.
- A generator performs no work while it is suspended.
- A slow operation before the next `yield` blocks the consumer on that same thread.

The behavior is similar to **cooperative control transfer**, not operating-system thread scheduling. Python compiles a generator function into bytecode and retains its suspended frame; it does not generate a new function or source code after every `yield`.

Actual concurrency requires other mechanisms such as threads, processes, or asynchronous programming. An **async generator** uses `async def` and `yield` and is consumed with `async for`, but even async code does not automatically imply a separate thread.

In [ ]:
import threading

def show_generator_thread():
    print("GENERATOR thread:", threading.current_thread().name)
    yield "one value"

print("CONSUMER thread:", threading.current_thread().name)
thread_demo = show_generator_thread()
print("CONSUMER received:", next(thread_demo))
print("Both messages normally show the same thread name.")

## 9. Sending a value into a generator

Generators also provide `.send(value)`. A `yield` expression can receive that value when execution resumes. This is an advanced consumer API; many everyday generators need only `next()` and `for`.

A newly created generator must first be started with `next(generator)` or `generator.send(None)` before sending a non-`None` value.

In [ ]:
def cart_discount_receiver():
    discount = yield "ready"
    while discount is not None:
        print(f"GENERATOR received discount: {discount}%")
        discount = yield f"accepted {discount}%"

receiver = cart_discount_receiver()
print(next(receiver))             # prime/start the generator
print(receiver.send(10))          # yield receives 10
print(receiver.send(20))          # yield receives 20
receiver.close()                  # stop the generator cleanly

## 10. Generators as a lazy data pipeline

Generators can be connected so that each stage processes one item and passes it to the next stage. This avoids intermediate result lists.

In [ ]:
def orders():
    for order in [
        {"id": "ORD-1", "value": 80},
        {"id": "ORD-2", "value": 250},
        {"id": "ORD-3", "value": 500},
    ]:
        print("SOURCE fetched", order["id"])
        yield order

def high_value_orders(source, minimum=200):
    for order in source:
        if order["value"] >= minimum:
            print("FILTER accepted", order["id"])
            yield order

def order_labels(source):
    for order in source:
        yield f"{order['id']} = ₹{order['value']}"

pipeline = order_labels(high_value_orders(orders()))

for label in pipeline:
    print("CONSUMER:", label)

## 11. Important behavior and common mistakes

- Calling a generator function does not run its body; it creates a generator object.
- A generator normally runs only when a consumer asks for its next value.
- Generators are forward-only and usually cannot be restarted after exhaustion.
- To repeat the sequence, call the generator function again to create a fresh generator.
- Printing a generator object does not display its values; consume it with `next()`, a loop, or `list()`.
- `list(generator)` is convenient, but it removes the memory advantage by storing all remaining values.
- An exception raised inside a generator is delivered to the consumer.
- Do not expect parallel execution: ordinary generator code runs synchronously on the calling thread.

## 12. Summary

- A generator is an iterator created by a generator function or generator expression.
- Lists and dictionaries are iterable collections, not generators.
- `enumerate`, `zip`, `map`, `filter`, and `reversed` return lazy built-in iterators that consumers use similarly.
- Consumers can use `next()`, `for`, `list()`, and other iterator-based APIs with generators.
- `yield` returns one value and suspends the function while preserving its execution state.
- Resumption continues immediately after the previous `yield`.
- Completion becomes `StopIteration`.
- An ordinary generator uses the consumer's thread; it does not create concurrency or run in the background.

## 13. Quick practice

1. Write a generator that yields five even product IDs: `2, 4, 6, 8, 10`.
2. Manually consume its first two values with `next()`, then consume the remainder with a loop.
3. Create a generator expression that applies 10% tax to `[100, 250, 500]`.
4. Modify `random_odd_numbers()` so the consumer can choose a maximum odd value.
5. Add a `break` to the order pipeline and observe which source orders are never fetched.